In [0]:
# ============================================================
# feature_filtering.py
# ============================================================
# Goal:
# Reduce the feature space while preserving physically
# meaningful descriptors for ML modeling.
#
# Steps:
# 1. Load engineered feature matrix
# 2. Remove target leakage
# 3. Separate X and y
# 4. Variance filtering
# 5. Univariate feature scoring
# 6. Keep top-ranked features
# 7. Save filtered matrix + feature rankings
# ============================================================

import pandas as pd
import numpy as np

from sklearn.feature_selection import (
    VarianceThreshold,
    f_regression,
    mutual_info_regression
)

# ============================================================
# 1. LOAD DATA
# ============================================================

INPUT_FILE = "full_feature_matrix.csv"

df = pd.read_csv(INPUT_FILE)

print("Initial shape:", df.shape)

# ============================================================
# 2. REMOVE TARGET LEAKAGE
# ============================================================
# We predict log_hydrogen_production_rate.
# Therefore raw hydrogen_production_rate must NOT be used
# as an input feature.

if "hydrogen_production_rate" in df.columns:
    df = df.drop(columns=["hydrogen_production_rate"])

# ============================================================
# 3. DEFINE TARGET
# ============================================================

TARGET = "log_hydrogen_production_rate"

y = df[TARGET]

X = df.drop(columns=[TARGET])

# ============================================================
# 4. VARIANCE FILTERING
# ============================================================
# Remove very low-information features.

selector = VarianceThreshold(threshold=0.01)

X_var = selector.fit_transform(X)

selected_cols = X.columns[selector.get_support()]

X_var = pd.DataFrame(X_var, columns=selected_cols)

print("After variance filtering:", X_var.shape)

# ============================================================
# 5. FEATURE SCORING
# ============================================================
# Two complementary metrics:
#
# f_regression:
#   linear relationship with target
#
# mutual_info_regression:
#   nonlinear dependency with target
# ============================================================

f_scores, f_pvalues = f_regression(X_var, y)

mi_scores = mutual_info_regression(
    X_var,
    y,
    random_state=42
)

# ============================================================
# 6. BUILD FEATURE RANKING TABLE
# ============================================================

feature_scores = pd.DataFrame({
    "feature": X_var.columns,
    "f_score": f_scores,
    "p_value": f_pvalues,
    "mutual_information": mi_scores
})

# sort by mutual information
feature_scores = feature_scores.sort_values(
    by="mutual_information",
    ascending=False
)

print("\nTop features:")
print(feature_scores.head(20))

# ============================================================
# 7. SELECT TOP FEATURES
# ============================================================
# Keep top-N most informative descriptors.
# You can adjust this later.

TOP_N = 40

top_features = feature_scores.head(TOP_N)["feature"].tolist()

X_filtered = X_var[top_features]

# add target back
filtered_df = pd.concat(
    [X_filtered, y.reset_index(drop=True)],
    axis=1
)

print("\nFinal filtered shape:", filtered_df.shape)

# ============================================================
# 8. SAVE OUTPUTS
# ============================================================

filtered_df.to_csv(
    "filtered_feature_matrix.csv",
    index=False
)

feature_scores.to_csv(
    "feature_scores.csv",
    index=False
)

print("\nFiles saved:")
print("- filtered_feature_matrix.csv")
print("- feature_scores.csv")

Check if the final 40 feature contains the ones below: 
- electronegativity
- radius
- valence
- structural
- experimental

In [0]:
# load filtered features
filtered_df = pd.read_csv("filtered_feature_matrix.csv")

# feature list (excluding target)
features = [
    c for c in filtered_df.columns
    if c != "log_hydrogen_production_rate"
]

print("Number of selected features:", len(features))

# ---------------------------------------------------
# Define descriptor categories
# ---------------------------------------------------

categories = {
    "electronegativity": [],
    "atomic_radius": [],
    "valence": [],
    "structural": [],
    "bandgap": [],
    "experimental": [],
    "other": []
}

for feat in features:

    feat_lower = feat.lower()

    if "en_" in feat_lower or "negativity" in feat_lower:
        categories["electronegativity"].append(feat)

    elif "radius" in feat_lower:
        categories["atomic_radius"].append(feat)

    elif "valence" in feat_lower or "unfilled" in feat_lower:
        categories["valence"].append(feat)

    elif (
        "tolerance" in feat_lower
        or "octahedral" in feat_lower
        or "lattice" in feat_lower
    ):
        categories["structural"].append(feat)

    elif "bandgap" in feat_lower:
        categories["bandgap"].append(feat)

    elif feat_lower in [
        "particle_size",
        "radiation_source_intensity",
        "specific_surface_area",
        "catalyst_amount",
        "calcination_temperature",
        "calcination_time",
        "bandgap_energy"
    ]:
        categories["experimental"].append(feat)

    else:
        categories["other"].append(feat)

# ---------------------------------------------------
# Print categorized features
# ---------------------------------------------------

for cat, feats in categories.items():

    print("\n==============================")
    print(cat.upper())
    print("==============================")

    for f in feats:
        print(f)

Force keeping of important descriptors for ML: 
- bandgap energy 
- tolerance factor 
- octahedral factor

In [0]:
force_keep = [
    "bandgap_energy",
    "tolerance_factor",
    "octahedral_factor"
]

In [0]:
full_df = pd.read_csv("full_feature_matrix.csv")

[c for c in force_keep if c in full_df.columns]

In [0]:
filtered_df = pd.read_csv("filtered_feature_matrix.csv")

[c for c in force_keep if c in filtered_df.columns]

In [0]:
missing = [
    c for c in force_keep
    if c not in filtered_df.columns
]

filtered_df = pd.concat(
    [filtered_df, full_df[missing]],
    axis=1
)

In [0]:
[c for c in force_keep if c in filtered_df.columns]

In [0]:
filtered_df.to_csv(
    "filtered_feature_matrix.csv",
    index=False
)

In [0]:
display (filtered_df)